CELL 1 : imports + scraping 

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3

category_urls = {
    "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
    "Classics": "https://books.toscrape.com/catalogue/category/books/classics_6/index.html",
}

all_books_data = []

for category_name, url in category_urls.items():
    response = requests.get(url)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price_text = book.find("p", class_="price_color").text
        rating_text = book.find("p", class_="star-rating")["class"][1]
        availability_text = book.find("p", class_="instock availability").text.strip()

        book_data = {
            "title": title,
            "price": price_text,
            "star_rating": rating_text,
            "availability": availability_text,
            "category": category_name
        }
        all_books_data.append(book_data)

    print(f"{category_name}: scraped {len(books)} books")

print("\nGrand total:", len(all_books_data))

Travel: scraped 11 books
Mystery: scraped 20 books
Historical Fiction: scraped 20 books
Classics: scraped 19 books

Grand total: 70


Cell 2: Build the DataFrame

In [2]:
df = pd.DataFrame(all_books_data)
print(df.head())
print(df.dtypes)

                                               title   price star_rating  \
0                            It's Only the Himalayas  £45.17         Two   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...  £49.43        Four   
2  See America: A Celebration of Our National Par...  £48.87       Three   
3  Vagabonding: An Uncommon Guide to the Art of L...  £36.94         Two   
4                               Under the Tuscan Sun  £37.33       Three   

  availability category  
0     In stock   Travel  
1     In stock   Travel  
2     In stock   Travel  
3     In stock   Travel  
4     In stock   Travel  
title           str
price           str
star_rating     str
availability    str
category        str
dtype: object


Cell 3: Clean price → price_gbp

In [3]:
df["price_gbp"] = df["price"].str.replace("£", "", regex=False).astype(float)
print(df[["price", "price_gbp"]].head())
print(df["price_gbp"].dtype)

    price  price_gbp
0  £45.17      45.17
1  £49.43      49.43
2  £48.87      48.87
3  £36.94      36.94
4  £37.33      37.33
float64


Cell 4: Clean star_rating → rating

In [12]:
rating_map = {
    "One": 1, 
    "Two": 2, 
    "Three": 3, 
    "Four": 4, 
    "Five": 5
}
df["rating"] = df["star_rating"].map(rating_map)
print(df[["star_rating", "rating"]].head())
print(df["rating"].dtype)
print(df["rating"].isnull().sum())

  star_rating  rating
0         Two       2
1        Four       4
2       Three       3
3         Two       2
4       Three       3
int64
0


Cell 5: Clean availability → in_stock

In [13]:
df["in_stock"] = df["availability"].str.contains("In stock")

print(df[["availability", "in_stock"]].head())
print(df["in_stock"].dtype)
print(df["in_stock"].value_counts())

  availability  in_stock
0     In stock      True
1     In stock      True
2     In stock      True
3     In stock      True
4     In stock      True
bool
in_stock
True    70
Name: count, dtype: int64


Cell 6: Check for broken rows (median-imputation safety net)

In [6]:
broken_rows = df[df["price_gbp"].isnull() | df["rating"].isnull()]
print("Broken rows found:", len(broken_rows))

if len(broken_rows) > 0:
    df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
    df["rating"] = df["rating"].fillna(df["rating"].median())
    print("Filled missing values with median.")
else:
    print("No broken rows — nothing to fix.")

Broken rows found: 0
No broken rows — nothing to fix.


Cell 7: Convert to price_inr

In [7]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

print(df[["price_gbp", "price_inr"]].head())
print(df["price_inr"].dtype)

   price_gbp  price_inr
0      45.17   4765.435
1      49.43   5214.865
2      48.87   5155.785
3      36.94   3897.170
4      37.33   3938.315
float64


Cell 8: Create SQLite tables (with re-run safety)


In [14]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER REFERENCES categories(category_id)
)
""")

cursor.execute("DELETE FROM books")
cursor.execute("DELETE FROM categories")
conn.commit()

print("Tables created and cleared successfully")

Tables created and cleared successfully


Cell 9: Insert data into the tables

In [15]:
unique_categories = df["category"].unique()

for cat in unique_categories:
    cursor.execute("INSERT OR IGNORE INTO categories (category_name) VALUES (?)", (cat,))
conn.commit()
print("Inserted", len(unique_categories), "categories")

for _, row in df.iterrows():
    cursor.execute("SELECT category_id FROM categories WHERE category_name = ?", (row["category"],))
    category_id = cursor.fetchone()[0]
    cursor.execute("""
        INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (row["title"], row["price_gbp"], row["price_inr"], row["rating"], int(row["in_stock"]), category_id))
conn.commit()
print("Inserted", len(df), "books")

Inserted 4 categories
Inserted 70 books


In [ ]:
Cell 10: Run the 5 SQL queries

In [16]:
def run_query(description, query):
    print("\n---", description, "---")
    print("QUERY:", query.strip())
    cursor.execute(query)
    results = cursor.fetchall()
    for row in results:
        print(row)
    return results

q1 = run_query("Books priced above £40", "SELECT title, price_gbp FROM books WHERE price_gbp > 40")
q2 = run_query("Top 5 most expensive books", "SELECT title, price_gbp FROM books ORDER BY price_gbp DESC LIMIT 5")
q3 = run_query("Distinct category names", "SELECT DISTINCT category_name FROM categories")
q4 = run_query("Books priced between £20 and £30", "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 30")
q5 = run_query("Top 10 highest-rated books with category name (JOIN)", """
    SELECT books.title, books.rating, categories.category_name
    FROM books
    JOIN categories ON books.category_id = categories.category_id
    ORDER BY books.rating DESC
    LIMIT 10
    """)


--- Books priced above £40 ---
QUERY: SELECT title, price_gbp FROM books WHERE price_gbp > 40
("It's Only the Himalayas", 45.17)
('Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Beyond', 49.43)
('See America: A Celebration of Our National Parks & Treasured Sites', 48.87)
('A Summer In Europe', 44.34)
('A Year in Provence (Provence #1)', 56.88)
('Sharp Objects', 47.82)
('The Past Never Ends', 56.5)
('The Murder of Roger Ackroyd (Hercule Poirot #4)', 44.1)
('The Last Mile (Amos Decker #2)', 54.21)
('A Time of Torment (Charlie Parker #14)', 48.35)
('Murder at the 42nd Street Library (Raymond Ambler #1)', 54.36)
('Boar Island (Anna Pigeon #19)', 59.48)
("The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)", 52.3)
('Tipping the Velvet', 53.74)
('A Flight of Arrows (The Pathfinders #2)', 55.53)
('Glory over Everything: Beyond The Kitchen House', 45.84)
('The Last Painting of Sara de Vos', 55.55)
('The Guernsey Literary and Potato Peel Pie Society', 49.53)
('G

Cell 11: pandas read_sql + merge comparison

In [17]:
df_expensive = pd.read_sql("SELECT title, price_gbp FROM books ORDER BY price_gbp DESC LIMIT 5", conn)
df_categories = pd.read_sql("SELECT * FROM categories", conn)
print("Top 5 expensive (via pd.read_sql):")
print(df_expensive)

df_books_full = pd.read_sql("SELECT * FROM books", conn)
df_joined = pd.merge(df_books_full, df_categories, on="category_id")
df_joined_top10 = df_joined[["title", "rating", "category_name"]].sort_values("rating", ascending=False).head(10)
print("\nTop 10 highest-rated (via pd.merge):")
print(df_joined_top10)

print("\nOriginal SQL JOIN result (q5) for comparison:")
for row in q5:
    print(row)

Top 5 expensive (via pd.read_sql):
                              title  price_gbp
0     Boar Island (Anna Pigeon #19)      59.48
1                           Candide      58.63
2                       Animal Farm      57.22
3  A Year in Provence (Provence #1)      56.88
4               The Past Never Ends      56.50

Top 10 highest-rated (via pd.merge):
                                                title  rating  \
35                                       Mrs. Houdini       5   
33            A Flight of Arrows (The Pathfinders #2)       5   
28  What Happened on Beale Street (Secrets of the ...       5   
29  The Bachelor Girl's Guide to Murder (Herringfo...       5   
19             A Time of Torment (Charlie Parker #14)       5   
10                 1,000 Places to See Before You Die       5   
47                                       The Red Tent       5   
44                              The Passion of Dolssa       5   
46                             Voyager (Outlander #3)       